# Kimi Audio Few-Shot AD Detection

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"]     = "/root/autodl-tmp/LLM_Model"

import json
import tempfile
from pathlib import Path

import torch
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-04-03 11:47:37.239 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-03 11:47:37.241 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-04-03 11:47:38.309 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-04-03 11:47:38.479 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [3]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [4]:
PAIR_NUM = 3

SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)


def classify_audio(wav_path: Path, control_wavs: list, dementia_wavs: list) -> str:
    """Classify a single audio file with PAIR_NUM-shot examples per class."""
    messages = []
    for wav in control_wavs:
        messages.extend([
            {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a healthy control speaker:"},
            {"role": "user",      "message_type": "audio", "content": str(wav)},
            {"role": "assistant", "message_type": "text",  "content": "Control"},
        ])
    for wav in dementia_wavs:
        messages.extend([
            {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + "This is a dementia speaker:"},
            {"role": "user",      "message_type": "audio", "content": str(wav)},
            {"role": "assistant", "message_type": "text",  "content": "Dementia"},
        ])
    messages.extend([
        {"role": "user", "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user", "message_type": "audio", "content": str(wav_path)},
    ])
    _, text = model.generate(messages, output_type="text", max_new_tokens=256)
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [5]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result")


def get_examples(df, audio_dir, n=PAIR_NUM):
    """Pick the first n Control and first n Dementia samples as few-shot examples."""
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    examples = {}
    for ad_val, label in label_map.items():
        rows = df[df["ad"] == ad_val].iloc[:n]
        wavs, ids = [], []
        for _, row in rows.iterrows():
            matches = list(audio_dir.glob(f"{label}/{row['session_id']}.*"))
            wavs.append(ensure_wav(matches[0]))
            ids.append(row["session_id"])
        examples[label] = {"session_ids": ids, "wavs": wavs}
    print(f"  Few-shot examples: Control={examples['Control']['session_ids']}, Dementia={examples['Dementia']['session_ids']}")
    return examples


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    # Select few-shot examples from this dataset
    examples = get_examples(df, audio_dir)
    example_ids = set(examples["Control"]["session_ids"] + examples["Dementia"]["session_ids"])
    control_wavs  = examples["Control"]["wavs"]
    dementia_wavs = examples["Dementia"]["wavs"]

    predictions, skipped = [], 0

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        # Skip few-shot example samples
        if row["session_id"] in example_ids:
            skipped += 1
            continue
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]), control_wavs, dementia_wavs)
            pred = parse_prediction(raw)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            raw, pred = "OOM", None
        except Exception as e:
            raw, pred = str(e), None
        finally:
            torch.cuda.empty_cache()
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1) * 100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) * 100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [6]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt_origin", "Pitt_origin", "Pitt_origin_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt_origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

[Pitt-origin-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt_origin, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-origin-raw: 100%|██████████| 552/552 [18:11<00:00,  1.98s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-origin-raw.csv
[Pitt-origin-raw]
  Accuracy:    58.42%
  F1:          0.5414
  Control Acc: 77.08%
  Dementia Acc:43.79%
  Valid: 546/552  Skipped: 6


In [7]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
# audio_dir = PROJECT_ROOT / "data/raw/Lu"
# evaluate_dataset(csv, audio_dir, "Lu-raw")

In [8]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Demucs")

[Pitt-origin-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-Demucs, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-origin-Demucs: 100%|██████████| 552/552 [19:52<00:00,  2.16s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-origin-Demucs.csv
[Pitt-origin-Demucs]
  Accuracy:    63.19%
  F1:          0.6540
  Control Acc: 64.58%
  Dementia Acc:62.09%
  Valid: 546/552  Skipped: 6


In [9]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
# evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [10]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Denoiser")

[Pitt-origin-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-Denoiser, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-origin-Denoiser: 100%|██████████| 552/552 [16:52<00:00,  1.83s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-origin-Denoiser.csv
[Pitt-origin-Denoiser]
  Accuracy:    66.67%
  F1:          0.6840
  Control Acc: 69.58%
  Dementia Acc:64.38%
  Valid: 546/552  Skipped: 6


In [11]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
# evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [12]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-origin-FRCRN_SE")

[Pitt-origin-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-FRCRN_SE, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-origin-FRCRN_SE: 100%|██████████| 552/552 [16:53<00:00,  1.84s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-origin-FRCRN_SE.csv
[Pitt-origin-FRCRN_SE]
  Accuracy:    65.57%
  F1:          0.6867
  Control Acc: 63.33%
  Dementia Acc:67.32%
  Valid: 546/552  Skipped: 6


In [13]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
# evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [14]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-origin-MossFormer")

[Pitt-origin-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-MossFormer, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-origin-MossFormer: 100%|██████████| 552/552 [17:31<00:00,  1.90s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-origin-MossFormer.csv
[Pitt-origin-MossFormer]
  Accuracy:    67.03%
  F1:          0.7231
  Control Acc: 54.58%
  Dementia Acc:76.80%
  Valid: 546/552  Skipped: 6


In [15]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
# evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [16]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Resemble")

[Pitt-origin-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-Resemble, exists=True
  Few-shot examples: Control=['002-0', '002-1', '002-2'], Dementia=['001-0', '001-2', '003-0']


Pitt-origin-Resemble: 100%|██████████| 552/552 [20:18<00:00,  2.21s/it]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_fewshot_result/Pitt-origin-Resemble.csv
[Pitt-origin-Resemble]
  Accuracy:    65.20%
  F1:          0.6701
  Control Acc: 67.92%
  Dementia Acc:63.07%
  Valid: 546/552  Skipped: 6


In [17]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
# evaluate_dataset(csv, audio_dir, "Lu-Resemble")